# Step 3: Dataset statistics and test-count check

1. Recompute the proposal's Table I with the shared loader, to check that the loader reads the data the same way.
2. Find out why we count 2,741 test messages while Peskov reports 2,475.

In [1]:
import json
import pandas as pd
from data_loader import load_split, DATA_DIR

## 1. Table I

In [2]:
rows = []
for split in ["train", "val", "test"]:
    df = load_split(split)
    rows.append({
        "split": split,
        "games": df.game_id.nunique(),
        "messages": len(df),
        "lies": df.is_lie.sum(),
        "lie rate (%)": round(100 * df.is_lie.mean(), 1),
        "avg words / msg": round(df.text.str.split().str.len().mean(), 1),  # words = split on spaces
    })
table = pd.DataFrame(rows).set_index("split")
table

,games,messages,lies,lie rate (%),avg words / msg
split,,,,,
train,9,13132,591,4.5,20.8
val,1,1416,56,4.0,18.0
test,2,2741,240,8.8,19.5


**Check:** the numbers should match Table I in the proposal.

In [3]:
proposal = pd.DataFrame({
    "games":           [9, 1, 2],
    "messages":        [13132, 1416, 2741],
    "lies":            [591, 56, 240],
    "lie rate (%)":    [4.5, 4.0, 8.8],
    "avg words / msg": [20.8, 18.0, 19.5],
}, index=["train", "val", "test"])

assert (table == proposal).all().all(), "Table I does not match the proposal"
print("OK: Table I matches the proposal.")

OK: Table I matches the proposal.


**Check:** each game belongs to exactly one split, so test games are never seen in training.

In [4]:
games = {s: set(load_split(s).game_id) for s in ["train", "val", "test"]}
print(games)
assert not (games["train"] & games["val"] or games["train"] & games["test"] or games["val"] & games["test"])
print("OK: no game appears in two splits.")

{'train': {1, 2, 3, 5, 6, 7, 8, 9, 10}, 'val': {11}, 'test': {4, 12}}
OK: no game appears in two splits.


## 2. Test-count check (2,741 vs 2,475)

The loader keeps only the sender's label, so here we read the receiver's label from the raw file. Some messages have no receiver label (`"NOANNOTATION"`).

In [5]:
with open(DATA_DIR / "test.jsonl", encoding="utf-8") as f:
    receiver_labels = [x for line in f for x in json.loads(line)["receiver_labels"]]

n_total = len(receiver_labels)
n_no_receiver_label = sum(x == "NOANNOTATION" for x in receiver_labels)
print("test messages:              ", n_total)
print("without a receiver label:   ", n_no_receiver_label)
print("with a receiver label:      ", n_total - n_no_receiver_label)

test messages:               2741
without a receiver label:    266
with a receiver label:       2475


In [6]:
assert n_total - n_no_receiver_label == 2475
print("OK: 2,741 - 266 messages without a receiver label = 2,475, Peskov's number.")

OK: 2,741 - 266 messages without a receiver label = 2,475, Peskov's number.


## Summary

- Table I in the proposal is reproduced exactly with the shared loader.
- Peskov's 2,475 test messages are most likely the ones that have a **receiver** label. For our task (the sender's label, ACTUAL LIE) all 2,741 test messages count.
- Note for the report: Peskov's human baseline (58.1 / 22.5) uses receiver labels, so it is computed on those 2,475 messages, not on all 2,741.